# COCO val2017 — Exploratory Data Analysis

**Requirement 2:** examine the data — number of observations, number and type of features, statistical dependencies, and anomalies — with visualizations and commentary.

This notebook is intentionally thin: all analysis lives in `objdetect.data.eda` (so it is unit-tested), and the notebook just calls those functions and plots. Run `uv run python scripts/download_data.py` first to fetch the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from objdetect import eda
from objdetect.data.coco import download_coco_annotations, load_coco_api

download_coco_annotations()

sns.set_theme(style="whitegrid")
coco = load_coco_api()
annotations = eda.annotations_frame(coco)
annotations.head()

## 1. Observations and features

Each **observation** is one annotation (a labelled bounding box). The **features** per observation: image id, class name, supercategory, box position `(x, y)`, box size `(width, height)`, derived `area`, `aspect_ratio`, `relative_area`, the `iscrowd` flag, and the parent image dimensions.

In [ ]:
print(f"Images:       {annotations['image_id'].nunique():,}")
print(f"Annotations:  {len(annotations):,}")
print(f"Classes:      {annotations['category'].nunique()}")
annotations.describe(include='all').T

## 2. Class distribution — imbalance

COCO is heavily imbalanced: `person` dominates while many classes are rare. This matters for training (rare classes are under-learned) and for the everyday-context subset we choose for fine-tuning.

In [ ]:
distribution = eda.class_distribution(annotations)
plt.figure(figsize=(12, 6))
top = distribution.head(25)
sns.barplot(data=top, x='instances', y='category', hue='category', legend=False, palette='viridis')
plt.title('Instances per class (top 25)')
plt.xlabel('Annotated instances'); plt.ylabel('Class')
plt.show()
distribution.head(10)

## 3. Scene density — boxes per image

Most images hold a handful of objects, but a long tail of crowded scenes contains dozens — these stress small-object detection and NMS.

In [ ]:
per_image = eda.boxes_per_image(annotations)
plt.figure(figsize=(9, 5))
sns.histplot(per_image, bins=range(0, per_image.max() + 2), color='steelblue')
plt.title('Objects per image'); plt.xlabel('Boxes in image'); plt.ylabel('Images')
plt.show()
print(per_image.describe())

## 4. Object scale

A large fraction of objects are small (a few percent of the image area or less). Small objects are the hardest case and the main reason FPN and two-stage detectors exist.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(annotations['relative_area'].clip(upper=0.5), bins=50, color='indianred')
plt.title('Object size relative to image'); plt.xlabel('Box area / image area'); plt.ylabel('Objects')
plt.show()
print(f"Objects smaller than 1% of the image: {(annotations['relative_area'] < 0.01).mean():.1%}")

## 5. Statistical dependencies — class co-occurrence

Classes are not independent: objects that share a context appear together far more than chance (e.g. *cup* with *dining table*, *person* with almost everything). This is the 'context' in Common Objects in Context.

In [ ]:
cooc = eda.class_cooccurrence(annotations, top_n=15)
plt.figure(figsize=(10, 8))
sns.heatmap(cooc, cmap='magma', square=True)
plt.title('Class co-occurrence in the same image (top 15)')
plt.show()

## 6. Anomalies

The dataset contains annotations that need care: tiny degenerate boxes, extreme aspect ratios (occluded slivers), `iscrowd` group boxes (one box over many objects), and boxes that exceed the image bounds. Our `CocoSubsetDataset` drops crowd and degenerate boxes during training for exactly this reason.

In [ ]:
anomalies = eda.find_anomalies(annotations)
anomalies['anomaly'].value_counts()